# 🌩️ Climate Extremes Analysis (CDO + Python)
Welcome to the interactive notebook for calculating Climate Extremes.
We will calculate **TXx (Hottest Day)** and **Rx1day (Max 1-Day Rainfall)**.

First, let's import the necessary libraries.

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import subprocess
import os

# ---> SET YOUR FILE NAMES HERE <--- 
tasmax_file = "/mnt/sdb2/WCS-S2_Data/ObservationalData/CPC/tasmax/tasmax_global_daily_gridded_obs_cpc_1979-2025.nc"
pr_file = "/mnt/sdb2/WCS-S2_Data/ObservationalData/CPC/P" # Make sure to fix this path too!

## 1. Calculating TXx (Hottest Day of the Year)
We will use Python's `subprocess` module to run the **CDO** command directly from this notebook.
The command is: `cdo yearmax input.nc output.nc`
(Note: CDO calculates the absolute maximum of tasmax using `yearmax` rather than an ECA operator)

In [ ]:
txx_output = "txx_annual.nc"

# Run CDO command if the output doesn't exist yet
if not os.path.exists(txx_output):
    print(f"Running CDO to calculate TXx from {tasmax_file}...")
    try:
        subprocess.run(["cdo", "yearmax", tasmax_file, txx_output], check=True)
        print("CDO calculation complete!")
    except FileNotFoundError:
        print("Error: CDO is not installed or not in your PATH.")
    except subprocess.CalledProcessError as e:
        print(f"CDO Error: Make sure {tasmax_file} exists in the same folder!")
else:
    print(f"{txx_output} already exists. Skipping calculation.")

## 2. Visualizing TXx with xarray and Cartopy
Now that CDO has done the heavy lifting, let's load the data and plot the mean Hottest Day across the dataset's time period.

In [ ]:
# Load the CDO output
try:
    ds_txx = xr.open_dataset(txx_output)
    
    # Find the correct variable (ignores CDO's 'time_bnds')
    var_name = [v for v in ds_txx.data_vars if 'lat' in ds_txx[v].dims][0]
    txx = ds_txx[var_name]
    
    # Calculate the average TXx over all years
    txx_mean = txx.mean(dim="time")
    
    # Plotting
    fig = plt.figure(figsize=(10, 6))
    ax = plt.axes(projection=ccrs.PlateCarree())
    
    ax.add_feature(cfeature.BORDERS, linestyle=":")
    ax.add_feature(cfeature.COASTLINE)
    
    txx_mean.plot(ax=ax, transform=ccrs.PlateCarree(), cmap="YlOrRd", cbar_kwargs={"label": "Temperature"})
    plt.title("Average Hottest Day of the Year (TXx)")
    plt.show()
except FileNotFoundError:
    print(f"Cannot find {txx_output}. Make sure the CDO command ran successfully.")

## 3. Calculating Rx1day (Max 1-Day Rainfall)
Let's repeat the process for extreme precipitation.

In [ ]:
rx1_output = "rx1day_annual.nc"

if not os.path.exists(rx1_output):
    print(f"Running CDO to calculate Rx1day from {pr_file}...")
    try:
        subprocess.run(["cdo", "eca_rx1day", pr_file, rx1_output], check=True)
        print("CDO calculation complete!")
    except subprocess.CalledProcessError:
        print(f"CDO Error: Make sure {pr_file} exists in the same folder!")
else:
    print(f"{rx1_output} already exists. Skipping calculation.")

In [ ]:
# Load and plot Rx1day
try:
    ds_rx1 = xr.open_dataset(rx1_output)
    
    # Find the correct variable (ignores CDO's 'time_bnds')
    var_name = [v for v in ds_rx1.data_vars if 'lat' in ds_rx1[v].dims][0]
    rx1 = ds_rx1[var_name]
    
    rx1_mean = rx1.mean(dim="time")
    
    fig = plt.figure(figsize=(10, 6))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.add_feature(cfeature.BORDERS, linestyle=":")
    ax.add_feature(cfeature.COASTLINE)
    
    rx1_mean.plot(ax=ax, transform=ccrs.PlateCarree(), cmap="Blues", cbar_kwargs={"label": "Precipitation (mm)"})
    plt.title("Average Max 1-Day Rainfall (Rx1day)")
    plt.show()
except FileNotFoundError:
    print(f"Cannot find {rx1_output}.")